# Install R and Packages

## Install R

In [1]:
%load_ext rpy2.ipython


## Install Packages

In [2]:
%%R
install.packages("nhanesA", repos="https://cloud.r-project.org")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
also installing the dependency ‘plyr’

trying URL 'https://cloud.r-project.org/src/contrib/plyr_1.8.9.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/nhanesA_1.4.1.tar.gz'

The downloaded source packages are in
	‘/tmp/Rtmphd88ZV/downloaded_packages’


# 1. NHANES Baseline Sample (2011-2012, 2013-2014)

## Pull Demographics files (DEMO_G, DEMO_H)

In [3]:
%%R

library(nhanesA)

cat("Starting pull: DEMO_G (2011-2012)\n")
demo_g <- nhanes("DEMO_G")
cat("Finished pull: DEMO_G\n\n")

cat("Starting pull: DEMO_H (2013-2014)\n")
demo_h <- nhanes("DEMO_H")
cat("Finished pull: DEMO_H\n\n")

Starting pull: DEMO_G (2011-2012)
Finished pull: DEMO_G

Starting pull: DEMO_H (2013-2014)
Finished pull: DEMO_H



## Make Columns stackable for both cohorts

In [4]:
%%R

cat("Adding missing columns\n")
all_cols <- union(names(demo_g), names(demo_h))

for (col in setdiff(all_cols, names(demo_g))) {
  demo_g[[col]] <- NA
}
for (col in setdiff(all_cols, names(demo_h))) {
  demo_h[[col]] <- NA
}

demo_g <- demo_g[, all_cols]
demo_h <- demo_h[, all_cols]
cat("Finished adding missing columns\n\n")

cat("Standardizing column types\n")
for (col in all_cols) {
  class_g <- class(demo_g[[col]])[1]
  class_h <- class(demo_h[[col]])[1]

  if (class_g != class_h) {
    cat("Type mismatch in", col, ":", class_g, "vs", class_h, "- converting both to character\n")
    demo_g[[col]] <- as.character(demo_g[[col]])
    demo_h[[col]] <- as.character(demo_h[[col]])
  }
}
cat("Finished standardizing column types\n\n")

Adding missing columns
Finished adding missing columns

Standardizing column types
Type mismatch in RIDEXAGY : numeric vs logical - converting both to character
Type mismatch in DMDHHSIZ : numeric vs factor - converting both to character
Type mismatch in DMDFMSIZ : numeric vs factor - converting both to character
Type mismatch in DMDHHSZA : numeric vs factor - converting both to character
Type mismatch in DMDHHSZB : numeric vs factor - converting both to character
Type mismatch in DMDHHSZE : numeric vs factor - converting both to character
Finished standardizing column types



## Stack the two cohorts

In [5]:
%%R
library(dplyr)

cat("Combining datasets\n")
demo <- bind_rows(demo_g, demo_h)
cat("Finished combining datasets\n")

cat("Rows:", nrow(demo), "\n")
cat("Columns:", ncol(demo), "\n")

Combining datasets
Finished combining datasets
Rows: 19931 
Columns: 48 



Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



## Get counts of Baseline Sample

In [6]:
%%R

cat("Rows:", nrow(demo), "\n")
cat("Columns:", ncol(demo), "\n")

Rows: 19931 
Columns: 48 


# 2. Age Restriction (50 < age < 80)

## Implement Age Restriction

In [7]:
%%R

cat("Checking missing values for RIDAGEYR\n")

missing_count <- sum(is.na(demo$RIDAGEYR))
total_count <- nrow(demo)

missing_percent <- (missing_count / total_count) * 100

cat("Missing RIDAGEYR count:", missing_count, "\n")
cat("Total rows:", total_count, "\n")
cat("Percent missing:", round(missing_percent, 2), "%\n")

Checking missing values for RIDAGEYR
Missing RIDAGEYR count: 0 
Total rows: 19931 
Percent missing: 0 %


In [8]:
%%R

cat("Applying age restriction: 50–80\n")

demo_age_restricted <- demo %>%
  dplyr::filter(RIDAGEYR >= 50 & RIDAGEYR <= 80)

Applying age restriction: 50–80


## Get counts of Age Restricted Sample

In [9]:
%%R

cat("Remaining participants:", nrow(demo_age_restricted), "\n")

Remaining participants: 5484 


# 3. Have Mortality Data

## Pull Mortality Data

In [10]:
%%R

library(dplyr)
library(readr)

cat("Starting all-cause mortality linkage pull...\n\n")

download_mortality_cycle <- function(file_name, cycle_label) {

  base_url <- "https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/linked_mortality/"
  url <- paste0(base_url, file_name)

  cat("Downloading:", url, "\n")

  mort <- read_fwf(
    file = url,
    fwf_cols(
      SEQN = c(1, 6),
      ELIGSTAT = c(15, 15),
      MORTSTAT = c(16, 16),
      PERMTH_EXM = c(46, 48)
    ),
    col_types = cols(
      SEQN = col_double(),
      ELIGSTAT = col_double(),
      MORTSTAT = col_double(),
      PERMTH_EXM = col_double()
    )
  ) %>%
    mutate(cycle = cycle_label)

  cat("Rows:", nrow(mort), "\n\n")

  return(mort)
}

mort_g <- download_mortality_cycle(
  "NHANES_2011_2012_MORT_2019_PUBLIC.dat",
  "2011-2012"
)

mort_h <- download_mortality_cycle(
  "NHANES_2013_2014_MORT_2019_PUBLIC.dat",
  "2013-2014"
)

mortality_all <- bind_rows(mort_g, mort_h) %>%
  transmute(
    SEQN,
    cycle,
    mortality_eligible = if_else(ELIGSTAT == 1, 1, 0),
    death = if_else(MORTSTAT == 1, 1, 0),
    followup_months = PERMTH_EXM,
    followup_years = PERMTH_EXM / 12
  )

cat("Total mortality rows:", nrow(mortality_all), "\n\n")

cat("Mortality eligibility counts:\n")
print(table(mortality_all$mortality_eligible, useNA = "ifany"))

cat("\nAll-cause death counts:\n")
print(table(mortality_all$death, useNA = "ifany"))

cat("\nFirst few rows:\n")
print(head(mortality_all))

# Optional: keep only mortality eligible people
mortality_eligible_all <- mortality_all %>%
  filter(mortality_eligible == 1)

cat("\nRows after keeping mortality eligible only:", nrow(mortality_eligible_all), "\n")

cat("\nAll-cause mortality summary among eligible people:\n")
print(
  mortality_eligible_all %>%
    count(death, name = "n") %>%
    mutate(percent = round(100 * n / sum(n), 1))
)

Starting all-cause mortality linkage pull...

Downloading: https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/linked_mortality/NHANES_2011_2012_MORT_2019_PUBLIC.dat 
Rows: 9756 

Downloading: https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/linked_mortality/NHANES_2013_2014_MORT_2019_PUBLIC.dat 
Rows: 10175 

Total mortality rows: 19931 

Mortality eligibility counts:

    0     1 
 7982 11949 

All-cause death counts:

    0     1  <NA> 
10854  1095  7982 

First few rows:
# A tibble: 6 × 6
   SEQN cycle     mortality_eligible death followup_months followup_years
  <dbl> <chr>                  <dbl> <dbl>           <dbl>          <dbl>
1 62161 2011-2012                  1     0              92           7.67
2 62162 2011-2012                  0    NA              NA          NA   
3 62163 2011-2012                  0    NA              NA          NA   
4 62164 2011-2012                  1     0              89           7.42
5 62165 2011-2012                  0    N

In addition: Warning messages:
1: One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat) 
2: One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat) 


## Restrict to those with Mortality

In [11]:
%%R

library(dplyr)

cat("Restricting demo_age_restricted to mortality eligible participants\n\n")

demo_age_restricted_mortality <- demo_age_restricted %>%
  inner_join(
    mortality_eligible_all,
    by = "SEQN"
  )

cat("Finished restriction\n\n")

cat("Rows in restricted dataset:",
    nrow(demo_age_restricted_mortality), "\n")

cat("Columns:",
    ncol(demo_age_restricted_mortality), "\n\n")

cat("Death counts:\n")
print(table(
  demo_age_restricted_mortality$death,
  useNA = "ifany"
))

cat("\nFollow-up summary:\n")
print(summary(
  demo_age_restricted_mortality$followup_years
))

Restricting demo_age_restricted to mortality eligible participants

Finished restriction

Rows in restricted dataset: 5472 
Columns: 53 

Death counts:

   0    1 
4495  977 

Follow-up summary:
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.     NAs 
  0.000   5.583   6.583   6.416   7.750   9.250     215 


# 4. Have Accelermeter Data

In [12]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


## Pull Hour Level Accelerometer Data

In [13]:
%%R

library(readr)

steps_hour <- read_csv("/content/drive/MyDrive/masters_thesis/NHANES-2-Cleaner/steps_hour.csv")

head(steps_hour)

Rows: 2432352 Columns: 4
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl (4): SEQN, PAXDAYM, HOUR, steps_in_hour

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
# A tibble: 6 × 4
   SEQN PAXDAYM  HOUR steps_in_hour
  <dbl>   <dbl> <dbl>         <dbl>
1 73557       2     0             0
2 73557       3     0             0
3 73557       4     0             0
4 73557       5     0             0
5 73557       6     0             0
6 73557       7     0             0


In [14]:
%%R

n_distinct(steps_hour$SEQN)

[1] 14676


## Restrict to Steps data

In [15]:
%%R

library(dplyr)

cat("Restricting to individuals with step data\n")

# get unique SEQN from steps
steps_ids <- steps_hour %>%
  distinct(SEQN)

demo_age_restricted_mortality_steps <- demo_age_restricted_mortality %>%
  inner_join(steps_ids, by = "SEQN")

cat("Finished restriction\n")
cat("Rows:", nrow(demo_age_restricted_mortality_steps), "\n")
cat("Columns:", ncol(demo_age_restricted_mortality_steps), "\n")

Restricting to individuals with step data
Finished restriction
Rows: 4625 
Columns: 53 


# Restricted Patients List

In [16]:
%%R

library(dplyr)

# keep only distinct SEQN values
seqn_table <- demo_age_restricted_mortality_steps %>%
  distinct(SEQN)

# save to CSV
write.csv(
  seqn_table,
  "/content/drive/MyDrive/masters_thesis/NHANES-2-Cleaner/Post_2-Restricted_Patients.csv",
  row.names = FALSE
)

cat("Saved file to:\n")
cat("/content/drive/MyDrive/masters_thesis/NHANES-2-Cleaner/Post_2-Restricted_Patients.csv\n")

Saved file to:
/content/drive/MyDrive/masters_thesis/NHANES-2-Cleaner/Post_2-Restricted_Patients.csv
